# P-PLP outpatient demo

This demo tries to predict whether patients with viral sinusitis will develop gingivitis within the defined time-at-risk window in an outpatient-style use case.

The notebook shows the end-to-end patient-level prediction workflow: inspect candidate concepts, build cohorts and labels, engineer patient-level features, and train a model.

Load the helper functions used throughout the demo for database access, cohort construction, feature engineering, and model training.

In [1]:
from p_plp.db import *
from p_plp.cohorts import *
from p_plp.feature_engineering import *
from p_plp.modeling import *

Create a database engine so the notebook can query the OMOP data and write intermediate tables.

In [2]:
engine = get_engine()

Inspect a few observed conditions and outcomes that are plausible for an outpatient scenario, then choose concept IDs for the target cohort and prediction outcome.

In [3]:
candidate_targets = list_observed_conditions(engine, limit=10)
candidate_outcomes = list_observed_outcomes(engine, limit=10)

candidate_targets, candidate_outcomes.head()

(   concept_id             concept_name vocabulary_id  n_occurrences
 0     4281516               Gingivitis        SNOMED            264
 1      133228            Dental caries        SNOMED            115
 2    40481087          Viral sinusitis        SNOMED            108
 3     4112343  Acute viral pharyngitis        SNOMED             73
 4      260139         Acute bronchitis        SNOMED             66
 5     4217975         Normal pregnancy        SNOMED             57
 6    37018196              Prediabetes        SNOMED             52
 7      320128   Essential hypertension        SNOMED             51
 8      439777                   Anemia        SNOMED             49
 9      436096             Chronic pain        SNOMED             33,
    concept_id             concept_name vocabulary_id  n_occurrences
 0     4281516               Gingivitis        SNOMED            264
 1      133228            Dental caries        SNOMED            115
 2    40481087          Viral sin

Set the outpatient demo configuration: the target and outcome concepts, the feature lookback window, the time-at-risk window, and the model type to train.

In [4]:
TARGET_CONCEPT_ID = 40481087   # Viral sinusitis
OUTCOME_CONCEPT_ID = 4281516   # Gingivitis
LOOKBACK_DAYS = 365
RISK_START_DAYS = 1
RISK_END_DAYS = 365
MODEL_NAME = "random_forest"

Build the target and outcome cohorts, then generate the patient labels based on whether the outcome happens within the configured time-at-risk window.

In [5]:
target_df = generate_target_cohort(engine, TARGET_CONCEPT_ID)
outcome_df = generate_outcome_cohort(engine, OUTCOME_CONCEPT_ID)
labels_df = generate_labels_time_at_risk(
    engine,
    risk_start_days=RISK_START_DAYS,
    risk_end_days=RISK_END_DAYS,
)

labels_df.head()

,subject_id,index_date,tar_start_date,tar_end_date,outcome_flag,outcome_date
0,63,1989-01-15,1989-01-16,1990-01-15,0,1987-07-05
1,117,1989-03-12,1989-03-13,1990-03-12,0,1986-09-21
2,86,1990-03-03,1990-03-04,1991-03-03,0,1995-08-05
3,66,1997-03-30,1997-03-31,1998-03-30,0,1989-12-17
4,82,1999-06-02,1999-06-03,2000-06-01,0,1999-05-12


Create the feature tables, assemble them into a single modeling dataset, and preview the resulting patient-level features with the outcome label.

In [6]:
build_demographic_features(engine)
build_prior_condition_count_features(engine, lookback_days=LOOKBACK_DAYS)
build_prior_drug_count_features(engine, lookback_days=LOOKBACK_DAYS)
build_prior_visit_count_features(engine, lookback_days=LOOKBACK_DAYS)
build_model_dataset(engine)

df_model = get_model_dataset(engine)
df_model.head()

,subject_id,index_date,age,gender_concept_id,n_prior_conditions,n_prior_drug_exposures,n_prior_visits,outcome_flag
0,63,1989-01-15,63,8507,4,40,7,0
1,117,1989-03-12,37,8532,1,4,3,0
2,86,1990-03-03,22,8507,0,0,1,0
3,66,1997-03-30,72,8507,0,116,109,0
4,82,1999-06-02,44,8532,1,0,2,0


Split the modeling dataset into predictors and labels, and identify which columns should be treated as numeric versus categorical.

In [7]:
X, y, num_cols, cat_cols = make_X_y(df_model)
X.head()

,age,gender_concept_id,n_prior_conditions,n_prior_drug_exposures,n_prior_visits
0,63,8507,4,40,7
1,37,8532,1,4,3
2,22,8507,0,0,1
3,72,8507,0,116,109
4,44,8532,1,0,2


Train the selected pipeline on the prepared dataset and evaluate its performance on the held-out test split.

In [8]:
model, X_test, y_test = train_pipeline(df_model, model_name=MODEL_NAME)
evaluate(model, X_test, y_test)

Accuracy: 0.9333
ROC-AUC: 1.0

              precision    recall  f1-score   support

           0      0.933     1.000     0.966        14
           1      0.000     0.000     0.000         1

    accuracy                          0.933        15
   macro avg      0.467     0.500     0.483        15
weighted avg      0.871     0.933     0.901        15



c:\Users\youri\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\youri\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\youri\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo